# Grass attempts at major catchments

In [ ]:
# launch this notebook from a GRASS GIS terminal - i.e. open grass and type Jupyter notebook into the console

from pathlib import Path

from grass.script.core import gisenv
import grass.script as gs
import grass.jupyter as gj
import geopandas as gpd
import pandas as pd

In [ ]:
import os
GRASS_RES = "/Applications/GRASS-8.4.app/Contents/Resources"
# Point to GRASS.app’s PROJ/GDAL data and neutralize conda’s
os.environ["PROJ_LIB"]  = f"{GRASS_RES}/share/proj"
os.environ["GDAL_DATA"] = f"{GRASS_RES}/share/gdal"
os.environ.pop("PROJ_DATA", None)  # if set by conda

In [ ]:
# gs.create_project("nbs_project", epsg=3448)
session = gj.init("nbs_project")


In [ ]:
dem = "/Users/robynhaggis/Documents/Geospatial_analysis/dem_filled.tif"   # r.watershed drainage-direction raster (EPSG:3448)
TARGET_AREA_KM2 = 40 #600                                 # "major basin" minimum area (tune)
STRAHLER_MIN    = 1       # <- add this (try 5; raise to 6 for fewer mains)
print(gs.read_command("g.proj", flags="p"))


In [ ]:
# 1) Import DEM & set region
gs.run_command("r.import", input=dem, output="dem_raw", overwrite=True)
gs.run_command("g.region", raster="dem_raw")

In [ ]:
# Compute DEM resolution to convert km² → cell threshold
info = gs.parse_command("r.info", map="dem_raw", flags="g")
res_m = float(info["nsres"])  # assumes square pixels
cells_threshold = int(round(TARGET_AREA_KM2 * 1_000_000 / (res_m**2)))
print(f"DEM res={res_m} m → threshold={cells_threshold} cells (~{TARGET_AREA_KM2} km²)")



In [ ]:
# 2) Derive flow fields at your chosen scale
gs.run_command("r.watershed",
               elevation="dem_raw",        # <- was dem_filled
               accumulation="accum",
               drainage="dir",
               threshold=cells_threshold,
               overwrite=True)

In [ ]:
# --- TUNE THE STREAM THRESHOLD FROM YOUR DATA ---
# Use a fraction of the actual max accumulation so we keep main stems + a few tribs
acc_stats = gs.parse_command("r.univar", map="accum", flags="g")
acc_max = float(acc_stats["max"])
cells_threshold_tuned = int(acc_max * 0.04)   # try 20% of max; adjust 0.10–0.30 if needed
print("acc_max =", acc_max, "  tuned_threshold =", cells_threshold_tuned)


In [ ]:
# Re-extract streams at the tuned threshold (new names to avoid confusion)
gs.run_command("r.stream.extract", elevation="dem_raw", accumulation="accum",
               threshold=cells_threshold_tuned, stream_raster="streams_tuned",
               direction="dir", overwrite=True)



In [ ]:
# (Optional) Order + auto-pick a sensible cut; else skip this block
gs.run_command("r.stream.order", stream_rast="streams_tuned", direction="dir",
               strahler="ord_tuned", overwrite=True)
ord_stats = gs.parse_command("r.univar", map="ord_tuned", flags="g")
ord_max = int(float(ord_stats["max"]))
print("ord_max =", ord_max)

if ord_max >= 3:
    STRAHLER_MIN = 1 #max(3, ord_max - 1)  # keep top orders
    print("Using STRAHLER_MIN =", STRAHLER_MIN)
    gs.mapcalc(f"streams_major = if(ord_tuned >= {STRAHLER_MIN}, streams_tuned, null())",
               overwrite=True)
else:
    # If the network still has no confluences, just use the tuned streams directly
    gs.run_command("g.copy", raster="streams_tuned,streams_major", overwrite=True)

# Sanity check
print(gs.read_command("r.univar", map="streams_major", flags="g"))


In [ ]:
# Basins per major system (last links only = river mouths)
gs.run_command("r.stream.basins", direction="dir", stream_rast="streams_major",
               basins="basins_major", flags="l", overwrite=True)

In [ ]:
# 1) Rebuild polygons (safe)
gs.run_command("r.to.vect", input="basins_major", output="basins_major_v",
               type="area", overwrite=True)

In [ ]:
# 2) Ensure every area has a category (cats live on centroids)
gs.run_command("v.category", input="basins_major_v", output="basins_major_v_cat",
               option="add", type="area", overwrite=True)

In [ ]:
# Give each basin a random color (for nicer viewing)
gs.run_command("r.colors", map="basins_major", color="random", overwrite=True)

# Export as GeoTIFF
out_tif = "/Users/robynhaggis/Documents/Geospatial_analysis/major_river_basins.tif"
gs.run_command("r.out.gdal", input="basins_major", output=out_tif,
               format="GTiff", createopt="COMPRESS=LZW", overwrite=True)
print("Wrote:", out_tif)

### stop here

In [ ]:
# Cell A — clean state & region
from grass.exceptions import CalledModuleError

# Remove any MASK if present (ignore error if none)
try:
    gs.run_command("r.mask", flags="r")
except CalledModuleError:
    pass

# Match processing region to your DEM
gs.run_command("g.region", raster="dem_raw", flags="p")

In [ ]:
# 0) Clean state
gs.run_command("r.mask", flags="r")                 # remove any MASK
gs.run_command("g.region", raster="dem_raw", flags="p")

# 1) Use YOUR dense stream network (threshold = 100 cells)
gs.run_command("r.stream.extract", elevation="dem_raw", accumulation="accum",
               threshold=100, stream_raster="streams_100",
               direction="dir", overwrite=True)

# 2) Basins for MOUTHS only (last links) -> should give island-wide mouth basins
gs.run_command("r.stream.basins", direction="dir",
               stream_rast="streams_100",
               basins="basins_mouths_100", flags="l", overwrite=True)

# 3) Check coverage; if any white remains, we’ll fill it
stats = gs.parse_command("r.univar", map="basins_mouths_100", flags="g")
print("null_cells =", stats["null_cells"])

# 4) (Only if gaps remain) Fill using all-links basins, assigned to the nearest mouth
if int(float(stats["null_cells"])) > 0:
    gs.run_command("r.stream.basins", direction="dir",
                   stream_rast="streams_100",
                   basins="basins_links_100", overwrite=True)  # no -l
    # Map each link-basin to the mouth-basin it overlaps most
    txt = gs.read_command("r.stats", flags="an",
                          input="basins_links_100,basins_mouths_100",
                          separator=",").strip().splitlines()
    from collections import defaultdict
    best = defaultdict(lambda: (0.0, None))  # link_id -> (overlap_area, mouth_id)
    for row in txt:
        link_id, mouth_id, area = row.split(",")
        L, M, A = int(link_id), int(mouth_id), float(area)
        if A > best[L][0]:
            best[L] = (A, M)
    rules = "\n".join(f"{L} = {M}" for L, (_, M) in best.items() if M is not None)
    path_rules = "/tmp/reclass_links2mouths.txt"
    open(path_rules, "w").write(rules)
    gs.run_command("r.reclass", input="basins_links_100",
                   output="basins_mouths_100_filled",
                   rules=path_rules, overwrite=True)
    FINAL = "basins_mouths_100_filled"
else:
    FINAL = "basins_mouths_100"

# 5) (Optional) Aggregate these mouth basins back to your existing big systems
#     If you have your previous "major-only" raster called basins_major, keep a copy:
gs.run_command("g.copy", raster="basins_major,basins_major_orig", overwrite=True)
#     Assign each mouth basin to the major it overlaps most:
txt = gs.read_command("r.stats", flags="an",
                      input=f"{FINAL},basins_major_orig",
                      separator=",").strip().splitlines()
from collections import defaultdict
best = defaultdict(lambda: (0.0, None))
for row in txt:
    fine, major, area = row.split(",")
    f, m, a = int(fine), int(major), float(area)
    if a > best[f][0]:
        best[f] = (a, m)
rules2 = "\n".join(f"{fid} = {mid}" for fid, (_, mid) in best.items() if mid is not None)
open("/tmp/reclass_pts2maj.txt", "w").write(rules2)
gs.run_command("r.reclass", input=FINAL,
               output="basins_major_from100_full",
               rules="/tmp/reclass_pts2maj.txt", overwrite=True)

# 6) Quick view/export
gs.run_command("r.colors", map="basins_major_from100_full", color="random", overwrite=True)
outdir = "/Users/robynhaggis/Documents/Geospatial_analysis"
gs.run_command("r.out.gdal", input="basins_major_from100_full",
               output=f"{outdir}/basins_major_from100_full.tif",
               format="GTiff", createopt="COMPRESS=LZW", overwrite=True)
print("Wrote:", f"{outdir}/basins_major_from100_full.tif")

In [ ]:
# ==== One cell: mouth-point basins with buffered coast + safe major merge ====
from grass.exceptions import CalledModuleError
from collections import defaultdict

# ---- tweakables ----
OUTDIR = "/Users/robynhaggis/Documents/Geospatial_analysis"
EDGE_GROW = 5    # <-- bigger buffer so streams ending just inland count as mouths
# --------------------

def raster_exists(name: str) -> bool:
    try:
        gs.parse_command("r.info", map=name, flags="g")
        return True
    except CalledModuleError:
        return False

def cat_first(token: str) -> int:
    """Parse r.stats category tokens like '12' or '12-12' safely."""
    return int(token.split("-")[0])

# 0) Clean state & region
try:
    gs.run_command("r.mask", flags="r")
except CalledModuleError:
    pass
gs.run_command("g.region", raster="dem_raw", flags="p")

# 1) Require flow direction
if not raster_exists("dir"):
    raise RuntimeError("Flow-direction raster 'dir' not found. Run r.watershed first (drainage='dir').")

# 2) Pick a stream raster if you already have one; else make a quick one (only to find mouths)
candidates = ["streams_w60", "streams_w100", "streams_100", "streams_tuned", "streams"]
STREAM_RAST = next((s for s in candidates if raster_exists(s)), None)
if STREAM_RAST is None:
    gs.run_command("r.watershed",
                   elevation="dem_raw",
                   accumulation="accum_tmp",
                   drainage="dir_tmp",
                   stream="streams_tmp",
                   threshold=60,
                   overwrite=True)
    STREAM_RAST = "streams_tmp"
print("Using stream raster:", STREAM_RAST)

# 3) Build an 'edge' raster using numeric rows/cols from region, then grow it
reg = gs.parse_command("g.region", flags="g")
nrows = int(reg["rows"]); ncols = int(reg["cols"])
gs.mapcalc(f"edge = if(col()==1 || col()=={ncols} || row()==1 || row()=={nrows}, 1, null())",
           overwrite=True)
edge_name = "edge"
if EDGE_GROW and EDGE_GROW > 0:
    gs.run_command("r.grow", input="edge", output="edge_buf", radius=EDGE_GROW, overwrite=True)
    edge_name = "edge_buf"

# 4) Mouth cells = stream pixels on (grown) border; to points; drop duplicates
gs.mapcalc(f"mouth_cells = if({STREAM_RAST} && {edge_name}, 1, null())", overwrite=True)
gs.run_command("r.to.vect", input="mouth_cells", output="mouth_pts", type="point", overwrite=True)
gs.run_command("v.extract", input="mouth_pts", output="mouth_pts_nodup",
               type="point", flags="d", overwrite=True)

# Export mouths to inspect in QGIS
gs.run_command("v.out.ogr", input="mouth_pts_nodup",
               output=f"{OUTDIR}/mouth_pts.gpkg", format="GPKG", overwrite=True)

# sanity: count mouths
try:
    counts = gs.parse_command("v.info", map="mouth_pts_nodup", flags="t")
    npts = int(counts.get("points", "0"))
except Exception:
    npts = 0
print("Mouth points found:", npts)
if npts == 0:
    raise RuntimeError("Still no mouth points. Increase EDGE_GROW (e.g., 10) and re-run this cell.")

# 5) Basins per mouth using ONLY flow direction (no stream_rast -> no mismatch)
gs.run_command("r.stream.basins", direction="dir", points="mouth_pts_nodup",
               basins="basins_mouths_pts", overwrite=True)

# 6) Fill tiny gaps to get wall-to-wall coverage
gs.run_command("r.grow.distance", input="basins_mouths_pts",
               value="basins_mouths_pts_full", overwrite=True)
nulls = gs.parse_command("r.univar", map="basins_mouths_pts_full", flags="g")["null_cells"]
print("Wall-to-wall mouth basins null_cells:", nulls)

# 7) (Optional) Merge to your existing big systems (if present), with safe parsing
if raster_exists("basins_major"):
    gs.run_command("g.copy", raster="basins_major,basins_major_orig", overwrite=True)

    txt = gs.read_command("r.stats", flags="an",
                          input="basins_mouths_pts_full,basins_major_orig",
                          separator=",").strip().splitlines()
    best = defaultdict(lambda: (0.0, None))  # mouth_id -> (overlap_area, major_id)

    for row in txt:
        parts = row.split(",")
        if len(parts) != 3:
            continue
        mouth_id, major_id, area = parts
        try:
            m = cat_first(mouth_id)
            M = cat_first(major_id)
            A = float(area)
        except ValueError:
            continue
        if A > best[m][0]:
            best[m] = (A, M)

    rules = "/tmp/reclass_mouth2major.txt"
    with open(rules, "w") as f:
        for m, (_, M) in best.items():
            if M is not None:
                f.write(f"{m} = {M}\n")

    gs.run_command("r.reclass", input="basins_mouths_pts_full",
                   output="basins_major_full_from_mouths",
                   rules=rules, overwrite=True)
    gs.run_command("r.colors", map="basins_major_full_from_mouths", color="random", overwrite=True)
    gs.run_command("r.out.gdal", input="basins_major_full_from_mouths",
                   output=f"{OUTDIR}/basins_major_full_from_mouths.tif",
                   format="GTiff", createopt="COMPRESS=LZW", overwrite=True)
    print("Wrote:", f"{OUTDIR}/basins_major_full_from_mouths.tif")
else:
    print("Note: 'basins_major' not found, skipping aggregation back to majors.")

# 8) Export wall-to-wall mouth basins (and the mouth points you can sanity-check)
gs.run_command("r.colors", map="basins_mouths_pts_full", color="random", overwrite=True)
gs.run_command("r.out.gdal", input="basins_mouths_pts_full",
               output=f"{OUTDIR}/basins_mouths_pts_full.tif",
               format="GTiff", createopt="COMPRESS=LZW", overwrite=True)
print("Wrote:", f"{OUTDIR}/basins_mouths_pts_full.tif")
# ==== end cell ====